In [1]:
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "Service": "bedrock.amazonaws.com"
      },
      "Action": [
        "s3:PutObject",
        "s3:GetObject",
        "s3:ListBucket"
      ],
      "Resource": [
        "arn:aws:s3:::rag-s3-dmac",
        "arn:aws:s3:::rag-s3-dmac/*"
      ]
    }
  ]
}

{'Version': '2012-10-17',
 'Statement': [{'Effect': 'Allow',
   'Principal': {'Service': 'bedrock.amazonaws.com'},
   'Action': ['s3:PutObject', 's3:GetObject', 's3:ListBucket'],
   'Resource': ['arn:aws:s3:::rag-s3-dmac', 'arn:aws:s3:::rag-s3-dmac/*']}]}

In [4]:
import boto3
import json
import time

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')

KB_ID = "8OOQBDOPXT"
MODEL_ID = "amazon.titan-multimodal-embed-g1-v1:0"  # or Sonnet

query = "Where does Alex Rivera live and what does he do?"

# Retrieve chunks (same as before)
bedrock_agent = boto3.client('bedrock-agent-runtime', region_name='us-east-1')

retrieve_response = bedrock_agent.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": query},
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "numberOfResults": 3
        }
    }
)

chunks = [ref['content']['text'] for ref in retrieve_response['retrievalResults']]
print("=== Retrieved Chunks ===")
for c in chunks:
    print("- " + c[:200] + "...\n")

# Augment prompt
context = "\n\n".join(chunks)
augmented_prompt = f"""Use only the following context to answer the question.
Do not refuse or apologize — provide the factual answer directly.

Context:
{context}

Question: {query}

Answer:"""

# Start Batch Converse job
OUTPUT_BUCKET = "rag-s3-dmac"  # ← replace with your bucket name
OUTPUT_PREFIX = "batch-rag-test-results/"

batch_job = bedrock_runtime.start_converse_batch(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": augmented_prompt}]}],
    inferenceConfig={"maxTokens": 512, "temperature": 0.0},
    outputConfig={
        "s3Output": {
            "bucket": OUTPUT_BUCKET,
            "keyPrefix": OUTPUT_PREFIX
        }
    }
)

job_id = batch_job['jobId']
print(f"Batch job started: {job_id}")
print(f"Results will be written to: s3://{OUTPUT_BUCKET}/{OUTPUT_PREFIX}job-{job_id}/")
print("Wait 5–15 minutes, then check S3 bucket or Bedrock console for status.")
print("To poll status, run: bedrock_runtime.get_converse_batch(jobId='{job_id}')")

=== Retrieved Chunks ===


AttributeError: 'BedrockRuntime' object has no attribute 'start_converse_batch'